# SMS Spam Classification using SVM (improved)
Dataset: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset

**What this notebook does**
1. Loads and de-duplicates the data
2. Explores it (class balance, length, word clouds)
3. Cleans text while *keeping* strong spam signals (URLs, phone numbers, money symbols)
4. Compares several models using cross-validation on the **training set only**
5. Tunes the SVM pipeline, then tunes the decision threshold on out-of-fold scores
6. Evaluates **once** on the untouched test set (confusion matrix, ROC, Precision-Recall)
7. Analyses errors, shows the most important words, saves the model and predicts new SMS

In [ ]:
# Install (if needed)
!pip -q install wordcloud nltk scikit-learn

In [ ]:
import os, re, string, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import nltk

from wordcloud import WordCloud
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import (train_test_split, GridSearchCV, StratifiedKFold,
                                     RepeatedStratifiedKFold, cross_validate,
                                     cross_val_score, cross_val_predict)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, roc_auc_score, precision_recall_curve,
                             average_precision_score, matthews_corrcoef)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

## 1. Load the dataset
Works in **Google Colab** (upload dialog) and locally (put `spam.csv` or `SMS.csv` next to the notebook).

In [ ]:
def find_csv():
    for name in ("spam.csv", "SMS.csv", "sms.csv"):
        if os.path.exists(name):
            return name
    found = glob.glob("*.csv")
    return found[0] if found else None

DATA_PATH = find_csv()
if DATA_PATH is None:
    try:
        from google.colab import files          # only exists inside Colab
        files.upload()
        DATA_PATH = find_csv()
    except ImportError:
        pass
assert DATA_PATH, "Dataset not found - place spam.csv (Kaggle) next to this notebook."

df = pd.read_csv(DATA_PATH, encoding="latin-1")
df = df[["v1", "v2"]].rename(columns={"v1": "label", "v2": "message"})
print("Loaded:", DATA_PATH)
df.head()

## 2. Data quality checks

In [ ]:
print("Shape:", df.shape)
print("Missing values:\n", df.isnull().sum(), sep="")
print("Exact duplicate rows:", df.duplicated().sum())
print()
print(df["label"].value_counts())

# Remove duplicates BEFORE the split so the same SMS can never be in both train and test
df = df.drop_duplicates().reset_index(drop=True)
print("\nAfter dropping duplicates:", df.shape)
print((df["label"].value_counts(normalize=True) * 100).round(2).astype(str) + " %")

## 3. Exploratory Data Analysis

In [ ]:
df["char_count"] = df["message"].str.len()
df["word_count"] = df["message"].str.split().str.len()

fig, ax = plt.subplots(1, 3, figsize=(16, 4))

sns.countplot(data=df, x="label", ax=ax[0])
for p in ax[0].patches:
    ax[0].annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2, p.get_height()),
                   ha="center", va="bottom")
ax[0].set_title("Class distribution")

sns.histplot(data=df, x="word_count", hue="label", bins=30, ax=ax[1])
ax[1].set_title("Words per message")

sns.boxplot(data=df, x="label", y="char_count", ax=ax[2])
ax[2].set_title("Characters per message")

plt.tight_layout(); plt.show()
print(df.groupby("label")[["char_count", "word_count"]].mean().round(1))

In [ ]:
spam_text = " ".join(df[df.label == "spam"].message)
ham_text  = " ".join(df[df.label == "ham"].message)

plt.figure(figsize=(14, 5))
for i, (title, txt) in enumerate([("Spam", spam_text), ("Ham", ham_text)], 1):
    plt.subplot(1, 2, i)
    plt.imshow(WordCloud(width=800, height=400, background_color="white").generate(txt))
    plt.axis("off"); plt.title(title, fontsize=14)
plt.tight_layout(); plt.show()

## 4. Preprocessing
Improvement: URLs, e-mails, phone numbers, numbers and money symbols are **replaced by tokens** instead of being deleted,
because things like "a long number", "a link" or "£" are strong spam indicators.

In [ ]:
stop_words = set(stopwords.words("english"))
lemm = WordNetLemmatizer()

URL_RE   = re.compile(r"(https?://\S+|www\.\S+)")
EMAIL_RE = re.compile(r"\S+@\S+")
LONGNUM  = re.compile(r"\d{5,}")      # phone numbers, short codes, prize codes
NUM      = re.compile(r"\d+")
MONEY    = re.compile(r"[£$€]")
PUNCT_TABLE = str.maketrans("", "", string.punctuation)

def clean_text(text):
    text = str(text).lower()
    text = URL_RE.sub(" urltoken ", text)
    text = EMAIL_RE.sub(" emailtoken ", text)
    text = MONEY.sub(" moneytoken ", text)
    text = LONGNUM.sub(" longnumtoken ", text)
    text = NUM.sub(" numtoken ", text)
    text = text.translate(PUNCT_TABLE)
    tokens = [lemm.lemmatize(w) for w in text.split() if w not in stop_words]
    return " ".join(tokens)

df["clean_message"] = df["message"].apply(clean_text)
df["target"] = df["label"].map({"ham": 0, "spam": 1})

df[["message", "clean_message", "target"]].head()

## 5. Train / Test split (stratified)
The test set is **not touched** until the final evaluation.

In [ ]:
X = df["clean_message"]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

print("Train:", X_train.shape, "| Spam in train: %.2f%%" % (y_train.mean() * 100))
print("Test :", X_test.shape,  "| Spam in test : %.2f%%" % (y_test.mean() * 100))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 6. Compare models (5-fold CV on the training set only)

In [ ]:
def make_pipe(clf):
    return Pipeline([("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)),
                     ("clf", clf)])

candidates = {
    "LinearSVC (SVM)":            LinearSVC(random_state=RANDOM_STATE, max_iter=10000),
    "LinearSVC balanced":         LinearSVC(class_weight="balanced", random_state=RANDOM_STATE, max_iter=10000),
    "Logistic Regression":        LogisticRegression(class_weight="balanced", max_iter=2000),
    "SGD (modified huber)":       SGDClassifier(loss="modified_huber", class_weight="balanced", random_state=RANDOM_STATE),
    "Multinomial Naive Bayes":    MultinomialNB(),
    "Complement Naive Bayes":     ComplementNB(),
}

rows = []
for name, clf in candidates.items():
    res = cross_validate(make_pipe(clf), X_train, y_train, cv=cv, n_jobs=-1,
                         scoring=["accuracy", "precision", "recall", "f1"])
    rows.append({"Model": name,
                 "Accuracy":  res["test_accuracy"].mean(),
                 "Precision": res["test_precision"].mean(),
                 "Recall":    res["test_recall"].mean(),
                 "F1":        res["test_f1"].mean()})

comparison = pd.DataFrame(rows).set_index("Model").sort_values("F1", ascending=False).round(4)
comparison

## 7. Hyperparameter tuning (SVM)
Optimised for **F1 of the spam class** (accuracy is misleading with ~13% spam). The grid also tunes the TF-IDF settings and class weights.

In [ ]:
svm_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True)),
    ("svm", LinearSVC(random_state=RANDOM_STATE, max_iter=10000)),
])

param_grid = {
    "tfidf__ngram_range":  [(1, 1), (1, 2)],
    "tfidf__min_df":       [1, 2],
    "tfidf__max_features": [None, 5000],
    "svm__C":              [0.1, 0.5, 1, 2, 5],
    "svm__class_weight":   [None, "balanced"],
}

grid = GridSearchCV(svm_pipe, param_grid, cv=cv, scoring="f1", n_jobs=-1, refit=True)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV F1     : %.4f" % grid.best_score_)

best_model = grid.best_estimator_
pd.DataFrame(grid.cv_results_).sort_values("rank_test_score")[
    ["rank_test_score", "mean_test_score", "std_test_score", "params"]].head(5)

## 8. Choose the decision threshold (without touching the test set)
LinearSVC predicts spam when its score is > 0. We pick the threshold that maximises F1 using **out-of-fold** scores on the training data.

In [ ]:
oof_scores = cross_val_predict(best_model, X_train, y_train, cv=cv, method="decision_function", n_jobs=-1)

prec, rec, thr = precision_recall_curve(y_train, oof_scores)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
best_idx = int(np.argmax(f1s))
BEST_THRESHOLD = float(thr[best_idx])

print("Default threshold : 0.000")
print("Tuned threshold   : %.3f  (OOF F1 = %.4f)" % (BEST_THRESHOLD, f1s[best_idx]))

plt.figure(figsize=(7, 4))
plt.plot(thr, f1s, label="F1")
plt.plot(thr, prec[:-1], label="Precision", alpha=.7)
plt.plot(thr, rec[:-1], label="Recall", alpha=.7)
plt.axvline(BEST_THRESHOLD, color="red", ls="--", label="chosen")
plt.axvline(0, color="grey", ls=":", label="default (0)")
plt.xlabel("Decision threshold"); plt.legend(); plt.title("Threshold tuning (train, out-of-fold)")
plt.show()

## 9. Final evaluation on the untouched test set

In [ ]:
test_scores = best_model.decision_function(X_test)
pred_default = (test_scores > 0).astype(int)
pred_tuned   = (test_scores > BEST_THRESHOLD).astype(int)

def summarize(name, y_true, y_pred, scores):
    return {"Model": name,
            "Accuracy":  accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred),
            "Recall":    recall_score(y_true, y_pred),
            "F1":        f1_score(y_true, y_pred),
            "MCC":       matthews_corrcoef(y_true, y_pred),
            "ROC-AUC":   roc_auc_score(y_true, scores),
            "PR-AUC":    average_precision_score(y_true, scores)}

results = pd.DataFrame([
    summarize("Tuned SVM (threshold 0)", y_test, pred_default, test_scores),
    summarize("Tuned SVM (tuned threshold)", y_test, pred_tuned, test_scores),
]).set_index("Model").round(4)
display(results)

print("Classification report (tuned threshold):")
print(classification_report(y_test, pred_tuned, target_names=["ham", "spam"], digits=4))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))

for a, (title, p) in zip(ax[:2], [("Threshold 0", pred_default), ("Tuned threshold", pred_tuned)]):
    ConfusionMatrixDisplay(confusion_matrix(y_test, p), display_labels=["ham", "spam"]).plot(
        ax=a, cmap="Blues", colorbar=False, values_format="d")
    a.set_title("Confusion matrix - " + title); a.grid(False)

fpr, tpr, _ = roc_curve(y_test, test_scores)
pp, rr, _ = precision_recall_curve(y_test, test_scores)
ax[2].plot(fpr, tpr, label="ROC (AUC = %.4f)" % roc_auc_score(y_test, test_scores))
ax[2].plot(rr, pp, label="PR (AP = %.4f)" % average_precision_score(y_test, test_scores))
ax[2].plot([0, 1], [0, 1], "k--", alpha=.4)
ax[2].set_xlabel("FPR / Recall"); ax[2].set_ylabel("TPR / Precision")
ax[2].set_title("ROC and Precision-Recall"); ax[2].legend(loc="lower right")
plt.tight_layout(); plt.show()

## 10. Stability check - repeated cross-validation on the full dataset

In [ ]:
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
for metric in ["accuracy", "precision", "recall", "f1"]:
    s = cross_val_score(best_model, X, y, cv=rcv, scoring=metric, n_jobs=-1)
    print(f"{metric:<10}: {s.mean():.4f} +/- {s.std():.4f}")

## 11. Error analysis - which messages does the model get wrong?

In [ ]:
err = df.loc[X_test.index, ["message"]].copy()
err["true"] = y_test.values
err["pred"] = pred_tuned
err["score"] = test_scores.round(3)

pd.set_option("display.max_colwidth", 140)
print("False positives (ham predicted as spam):", int(((err.true == 0) & (err.pred == 1)).sum()))
display(err[(err.true == 0) & (err.pred == 1)].sort_values("score", ascending=False).head(10))

print("False negatives (spam missed):", int(((err.true == 1) & (err.pred == 0)).sum()))
display(err[(err.true == 1) & (err.pred == 0)].sort_values("score").head(10))

## 12. Most important words

In [ ]:
tfidf = best_model.named_steps["tfidf"]
svm   = best_model.named_steps["svm"]
feat  = np.array(tfidf.get_feature_names_out())
coef  = svm.coef_.ravel()
order = np.argsort(coef)

top_spam = pd.Series(coef[order[-15:]][::-1], index=feat[order[-15:]][::-1])
top_ham  = pd.Series(coef[order[:15]], index=feat[order[:15]])

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
top_spam.sort_values().plot.barh(ax=ax[0], color="crimson"); ax[0].set_title("Top spam indicators")
top_ham.sort_values(ascending=False).plot.barh(ax=ax[1], color="seagreen"); ax[1].set_title("Top ham indicators")
plt.tight_layout(); plt.show()

## 13. Save the model and predict new SMS

In [ ]:
joblib.dump({"model": best_model, "threshold": BEST_THRESHOLD}, "sms_spam_svm.joblib")
print("Saved -> sms_spam_svm.joblib")

def predict_sms(text, model=best_model, threshold=BEST_THRESHOLD):
    score = float(model.decision_function([clean_text(text)])[0])
    label = "Spam" if score > threshold else "Ham"
    return label, round(score, 3)

samples = [
    "Congratulations! You have won a free iPhone. Click here now.",
    "Hi, are we meeting tomorrow at 3 PM?",
    "URGENT! Your mobile number has won a 2000 pound prize. Call 09061701461 now",
    "Can you pick up some milk on your way home?",
    "FREE entry to our weekly competition! Text WIN to 80086 to enter",
]
pd.DataFrame([(s, *predict_sms(s)) for s in samples], columns=["message", "prediction", "score"])